<a href="https://colab.research.google.com/github/InesAnindiyta/inesanin/blob/master/bigdata3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

pengenalan spark dataframe

In [22]:
# Contoh membuat DataFrame sederhana dan operasi dasar
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('HandsOnPertemuan3').getOrCreate()

data = [('Jeck', 'Sales', 3000),
        ('Michael', 'Sales', 4100),
        ('Mawar', 'CEO', 5000),
        ('Rina', 'Sekertaris', 43000)]
columns = ['nama', 'departemen', 'gaji']

df = spark.createDataFrame(data, schema=columns)
df.show()

+-------+----------+-----+
|   nama|departemen| gaji|
+-------+----------+-----+
|   Jeck|     Sales| 3000|
|Michael|     Sales| 4100|
|  Mawar|       CEO| 5000|
|   Rina|Sekertaris|43000|
+-------+----------+-----+



Transformasi Dasar dengan DataFrames

In [23]:
df.select('Nama', 'Gaji').show()
df.filter(df['Gaji'] > 3000).show()
df.groupBy('Departemen').avg('Gaji').show()


+-------+-----+
|   Nama| Gaji|
+-------+-----+
|   Jeck| 3000|
|Michael| 4100|
|  Mawar| 5000|
|   Rina|43000|
+-------+-----+

+-------+----------+-----+
|   nama|departemen| gaji|
+-------+----------+-----+
|Michael|     Sales| 4100|
|  Mawar|       CEO| 5000|
|   Rina|Sekertaris|43000|
+-------+----------+-----+

+----------+---------+
|Departemen|avg(Gaji)|
+----------+---------+
|     Sales|   3550.0|
|       CEO|   5000.0|
|Sekertaris|  43000.0|
+----------+---------+



Bekerja dengan Tipe Data Kompleks

In [24]:
# Contoh manipulasi tipe data kompleks
df.withColumn('Bonus', df['gaji'] * 0.1).show()
df.withColumn('Total', df['gaji'] + df['gaji']).show()

+-------+----------+-----+------+
|   nama|departemen| gaji| Bonus|
+-------+----------+-----+------+
|   Jeck|     Sales| 3000| 300.0|
|Michael|     Sales| 4100| 410.0|
|  Mawar|       CEO| 5000| 500.0|
|   Rina|Sekertaris|43000|4300.0|
+-------+----------+-----+------+

+-------+----------+-----+-----+
|   nama|departemen| gaji|Total|
+-------+----------+-----+-----+
|   Jeck|     Sales| 3000| 6000|
|Michael|     Sales| 4100| 8200|
|  Mawar|       CEO| 5000|10000|
|   Rina|Sekertaris|43000|86000|
+-------+----------+-----+-----+



Operasi Data Lanjutan

In [25]:
# Contoh menggunakan window functions
from pyspark.sql.window import Window
from pyspark.sql import functions as F

windowSpec = Window.partitionBy('Departemen').orderBy('gaji')
df.withColumn('Rank', F.rank().over(windowSpec)).show()

+-------+----------+-----+----+
|   nama|departemen| gaji|Rank|
+-------+----------+-----+----+
|  Mawar|       CEO| 5000|   1|
|   Jeck|     Sales| 3000|   1|
|Michael|     Sales| 4100|   2|
|   Rina|Sekertaris|43000|   1|
+-------+----------+-----+----+



 Kesimpulan dan Eksplorasi Lebih Lanjut

In [26]:
!pip install kaggle

!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [29]:
df = spark.read.csv('airlines_flights_data.csv', header=True, inferSchema=True)
df.printSchema()
df.show()

root
 |-- index: integer (nullable = true)
 |-- airline: string (nullable = true)
 |-- flight: string (nullable = true)
 |-- source_city: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stops: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- destination_city: string (nullable = true)
 |-- class: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- days_left: integer (nullable = true)
 |-- price: integer (nullable = true)

+-----+---------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|index|  airline| flight|source_city|departure_time|stops| arrival_time|destination_city|  class|duration|days_left|price|
+-----+---------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|    0| SpiceJet|SG-8709|      Delhi|       Evening| zero|        Night|          Mumbai|Economy|    2.17|        1| 5953|
|    1| Spic

In [32]:
from pyspark.sql.functions import col, avg, min, max

# Filtering tiket mahal
df_expensive = df.filter(col("price") > 5955)
df_expensive.show(10)

# Agregasi: rata-rata, min, max
df_avg_price = df.groupBy("class").agg(avg("price").alias("average_price"))
df_avg_price.show(10)

df_min_price = df.groupBy("class").agg(min("price").alias("smallest_price"))
df_min_price.show(10)

df_max_price = df.groupBy("class").agg(max("price").alias("highest_price"))
df_max_price.show(10)

+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|index| airline| flight|source_city|departure_time|stops| arrival_time|destination_city|  class|duration|days_left|price|
+-----+--------+-------+-----------+--------------+-----+-------------+----------------+-------+--------+---------+-----+
|    2| AirAsia| I5-764|      Delhi| Early_Morning| zero|Early_Morning|          Mumbai|Economy|    2.17|        1| 5956|
|    6| Vistara| UK-927|      Delhi|       Morning| zero|      Morning|          Mumbai|Economy|    2.08|        1| 6060|
|    7| Vistara| UK-951|      Delhi|     Afternoon| zero|      Evening|          Mumbai|Economy|    2.17|        1| 6060|
|   24|  Indigo|6E-5328|      Delhi|       Morning| zero|      Morning|          Mumbai|Economy|     2.5|        1| 6165|
|   25| Vistara| UK-933|      Delhi|     Afternoon| zero|      Evening|          Mumbai|Economy|    2.17|        1| 6690|
|   26|  Indigo|6E-2046|

In [35]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

windowSpec = Window.partitionBy("class").orderBy(col("price").desc())

df_peringkat = df.withColumn("rankings", rank().over(windowSpec))
df_peringkat.show(10)

+------+-------+------+-----------+--------------+-----------+------------+----------------+--------+--------+---------+------+--------+
| index|airline|flight|source_city|departure_time|      stops|arrival_time|destination_city|   class|duration|days_left| price|rankings|
+------+-------+------+-----------+--------------+-----------+------------+----------------+--------+--------+---------+------+--------+
|261377|Vistara|UK-772|    Kolkata|       Morning|        one|       Night|           Delhi|Business|    13.5|        3|123071|       1|
|216096|Vistara|UK-811|      Delhi| Early_Morning|two_or_more|     Evening|         Kolkata|Business|   10.92|        5|117307|       2|
|215859|Vistara|UK-809|      Delhi|       Evening|two_or_more|     Evening|         Kolkata|Business|   21.08|        1|116562|       3|
|277345|Vistara|UK-870|  Hyderabad|         Night|        one|   Afternoon|          Mumbai|Business|   16.42|        3|115211|       4|
|270999|Vistara|UK-772|    Kolkata|      